# ROGII Wellbore Geology — Honest Physics-Signal Stack (inference)

Target-free, no leak, no blending. One LightGBM (trained offline on 773 wells,
5-fold GroupKFold CV row-RMSE **10.62** vs PF 10.82, constant 15.9) over
domain-physics signals: particle filter, beam search, multi-scale NCC, Q-3D
tortuosity, relative trajectory, typewell-GR residuals. Predicts
`ΔTVT = TVT − last_known_TVT` for the hidden tail; only observed test logs used.

In [ ]:
import os

In [ ]:
"""
Particle-filter geosteering tracker (numpy, vectorised over particles).

State per particle: (pos, rate) where pos = TVT + Z  ("structural" position that
is ~constant when geology is flat), rate = d(pos)/dMD. At each eval step we
propagate with momentum, convert to TVT = pos - Z, score against the typewell
GR(TVT) curve via a Gaussian likelihood, resample, and take the weighted-mean TVT.

Predictions are made only for the eval rows (TVT_input is NaN). A seed ensemble
is combined with weights ∝ exp(loglik/scale). This is the core signal that gets
geosteering solutions to single-digit RMSE.
"""
import numpy as np

# tuned defaults (close to strong public solutions)
MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5


def _run_pf(md, z, gr, tw_tvt, tw_gr, last_tvt, last_Z, ir, gs,
            n_particles=500, seed=0):
    rng = np.random.default_rng(seed)
    N = n_particles
    ls = last_tvt + last_Z
    pos = ls + 2.0 * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N) / N
    lo, hi = tw_tvt[0] - 100, tw_tvt[-1] + 100
    out = np.empty(len(md)); loglik = 0.0
    prev_md = md[0] - 1.0
    for i in range(len(md)):
        dm = max(md[i] - prev_md, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dm + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - z[i], lo, hi)
        pos = tvt_p + z[i]
        if np.isfinite(gr[i]):
            eg = np.interp(tvt_p, tw_tvt, tw_gr)
            d = (gr[i] - eg) / gs
            lk = np.maximum(np.exp(-0.5 * np.minimum(d * d, 600.0)), 1e-300)
            avg = float((w * lk).sum()); loglik += np.log(max(avg, 1e-300))
            w = w * lk; s = w.sum(); w = w / s if s > 0 else np.ones(N) / N
        neff = 1.0 / (w * w).sum()
        if neff < RESAMP * N:
            cum = np.cumsum(w); u0 = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos = pos[idx] + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w = np.ones(N) / N
        out[i] = float(np.dot(w, pos - z[i]))
        prev_md = md[i]
    return out, loglik


def pf_predict(hw, tw, n_particles=500, n_seeds=48, scale=5.0):
    """Return a full-length TVT array (known rows kept, eval rows predicted)."""
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)

    kn = hw[hw["TVT_input"].notna()]
    ev = hw[hw["TVT_input"].isna()]
    out = hw["TVT_input"].to_numpy(float).copy()
    if len(ev) == 0:
        return out

    last = kn.iloc[-1]
    last_tvt = float(last["TVT_input"]); last_Z = float(last["Z"])
    # GR noise sigma from known section
    tw_at_k = np.interp(kn["TVT_input"].to_numpy(), tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn["GR"].fillna(0).to_numpy() - tw_at_k), 10., 60.))
    # initial structural rate from last 30 known points
    tail = kn.tail(30)
    dt = np.diff(tail["TVT_input"].to_numpy()); dz = np.diff(tail["Z"].to_numpy())
    dmv = np.diff(tail["MD"].to_numpy()); m = dmv > 0
    ir = float(np.median((dt + dz)[m] / dmv[m])) if m.sum() >= 3 else 0.0

    gr_all = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    md = ev["MD"].to_numpy(float); z = ev["Z"].to_numpy(float)
    gr = gr_all[ev.index.to_numpy()]
    # prepend last known so momentum starts from the right place
    md = np.concatenate([[float(last["MD"])], md])
    z = np.concatenate([[last_Z], z])
    gr = np.concatenate([[np.nan], gr])

    preds = np.empty((n_seeds, len(md))); liks = np.empty(n_seeds)
    for s in range(n_seeds):
        preds[s], liks[s] = _run_pf(md, z, gr, tw_tvt, tw_gr, last_tvt, last_Z,
                                    ir, gs, n_particles, seed=s)
    liks -= liks.max()
    wts = np.exp(liks / scale); wts /= wts.sum()
    est = (wts[:, None] * preds).sum(0)[1:]        # drop the prepended known row
    out[ev.index.to_numpy()] = est
    return out


In [ ]:
"""Beam-search geosteering tracker (numba JIT). Complements the particle filter:
finds a low-cost monotone-ish path through the typewell GR(TVT) grid that matches
the horizontal GR signature. Ensembled over several stiffness configs."""
import numpy as np
from numba import njit
from scipy.signal import savgol_filter

# (beam_size, move_cost, err_scale, savgol_radius)
BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2), (10, 8.0, 64.0, 2), (8, 35.0, 220.0, 1),
    (10, 14.0, 90.0, 5), (20, 4.0, 36.0, 3), (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
]


@njit(cache=False)
def _beam(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS * 6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.0; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt:
                    continue
                tot = cost + (gv - tw_gr[ni]) ** 2 / es + mc * (d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni:
                        fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]:
                        cC[fnd] = tot; cP[fnd] = bi
                elif nc < MAX:
                    cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i + 1, nc):
                if cC[j] < cC[mi]:
                    mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]
                cC[i], cC[mi] = cC[mi], cC[i]
                cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]:
            best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n - 1, -1, -1):
        path[s] = hI[s, b]; b = hP[s, b]
    return path


def beam_predict(hw, tw):
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    out = hw["TVT_input"].to_numpy(float).copy()
    if len(ev) == 0:
        return out
    last_tvt = float(kn.iloc[-1]["TVT_input"])
    gr_all = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    hgr = gr_all[ev.index.to_numpy()]
    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))
    paths = []
    for BS, mc, es, r in BEAM_CONFIGS:
        if r > 0 and len(hgr) > max(3, 2 * r + 1):
            win = min(2 * r + 1, len(hgr) if len(hgr) % 2 == 1 else len(hgr) - 1)
            sgr = savgol_filter(hgr, win, min(2, win - 1))
        else:
            sgr = hgr.copy()
        paths.append(tw_tvt[_beam(sgr.astype(np.float64), tw_gr, si, BS, float(mc), float(es))])
    out[ev.index.to_numpy()] = np.stack(paths, 0).mean(0)
    return out


In [ ]:
def ps_index(h):
    return int(h['TVT_input'].notna().sum())

"""
Compute honest domain signals per well (post-PS rows) for the LightGBM residual
stack. Target-free (no hidden TVT used). Saves parquet for train and test.

Signals: PF path (delta), beam path (delta), multi-scale NCC vs typewell,
self-correlation vs the lateral's own known section, Q-3D tortuosity, relative
trajectory geometry, GR texture, typewell GR residuals at PF/anchor baselines.
Target: dTVT = TVT - last_known_TVT (train only).
"""
import sys, os, glob, time
import numpy as np
import pandas as pd
from joblib import Parallel, delayed






ANCH_OFF = [-20, -10, -5, 0, 5, 10, 20]


def multiscale_ncc(kgr, ktvt, hgr, hws=(8, 15, 25), stride=3):
    out = []
    for hw in hws:
        win = 2 * hw + 1; nk = len(kgr); nh = len(hgr)
        if nk < win + 1 or nh == 0:
            out.append((np.full(nh, ktvt[-1] if len(ktvt) else 0.0, np.float32), np.zeros(nh, np.float32)))
            continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().to_numpy(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().to_numpy(np.float32)
        sts = np.arange(0, nk - win + 1, stride, dtype=np.int32); M = len(sts)
        if M == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32))); continue
        C = kg[sts[:, None] + np.arange(win, dtype=np.int32)[None, :]]
        Cn = (C - C.mean(1, keepdims=True)) / (C.std(1, keepdims=True) + 1e-6)
        hp = np.pad(hg, hw, mode="edge")
        Hm = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]]
        Hn = (Hm - Hm.mean(1, keepdims=True)) / (Hm.std(1, keepdims=True) + 1e-6)
        ncc = Hn @ Cn.T / win
        best = ncc.argmax(1); score = ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best] + hw, 0, nk - 1)].astype(np.float32), score))
    return out


def tortuosity(x, y, z, md, win=50):
    d = np.stack([np.gradient(x), np.gradient(y), np.gradient(z)], 1)
    dm = np.gradient(md)[:, None] + 1e-9
    t = d / dm
    n = np.linalg.norm(t, axis=1, keepdims=True) + 1e-9
    u = t / n
    kappa = np.linalg.norm(np.gradient(u, axis=0), axis=1)   # curvature proxy
    return pd.Series(kappa).rolling(win, min_periods=1).sum().to_numpy(np.float32)


def build_well(path, is_train):
    wid = os.path.basename(path).split("__")[0]
    base = os.path.dirname(path)
    try:
        h = pd.read_csv(path)
        tw = pd.read_csv(f"{base}/{wid}__typewell.csv")
    except Exception:
        return None
    ps = ps_index(h)
    if ps < 10 or ps >= len(h) - 3:
        return None
    if is_train and "TVT" not in h.columns:
        return None
    tw_s = tw.sort_values("TVT"); tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)
    if len(tw_tvt) < 3:
        return None

    tps = float(h["TVT_input"].iloc[ps - 1])
    md = h["MD"].to_numpy(float); x = h["X"].to_numpy(float); y = h["Y"].to_numpy(float); z = h["Z"].to_numpy(float)
    gr_all = h["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    ev = np.arange(ps, len(h)); nev = len(ev)

    pf = pf_predict(h, tw, n_particles=300, n_seeds=24, scale=12.0)
    bm = beam_predict(h, tw)
    ktvt = h["TVT_input"].to_numpy(float)[:ps]
    kgr = gr_all[:ps]; hgr = gr_all[ps:]
    ncc = multiscale_ncc(kgr, ktvt, hgr)
    tort = tortuosity(x, y, z, md)

    md_ps, x_ps, y_ps, z_ps = md[ps - 1], x[ps - 1], y[ps - 1], z[ps - 1]
    gr_s = pd.Series(gr_all)
    f = {
        "well": wid, "id": [f"{wid}_{i}" for i in ev],
        "last_known_tvt": tps,
        "pf_d": (pf[ev] - tps).astype(np.float32),
        "beam_d": (bm[ev] - tps).astype(np.float32),
        "pf_vs_beam": (pf[ev] - bm[ev]).astype(np.float32),
        "ncc8_d": np.clip(ncc[0][0] - tps, -80, 80), "ncc8_s": ncc[0][1],
        "ncc15_d": np.clip(ncc[1][0] - tps, -80, 80), "ncc15_s": ncc[1][1],
        "ncc25_d": np.clip(ncc[2][0] - tps, -80, 80), "ncc25_s": ncc[2][1],
        "ncc_mean_d": np.clip((ncc[0][0] + ncc[1][0] + ncc[2][0]) / 3 - tps, -80, 80),
        "tort": tort[ev],
        "d_md": (md[ev] - md_ps).astype(np.float32),
        "d_z": (z[ev] - z_ps).astype(np.float32),
        "d_xy": np.hypot(x[ev] - x_ps, y[ev] - y_ps).astype(np.float32),
        "dz_dmd": (np.gradient(z) / (np.gradient(md) + 1e-9))[ev].astype(np.float32),
        "gr": gr_all[ev].astype(np.float32),
        "gr_m21": gr_s.rolling(21, center=True, min_periods=1).mean().to_numpy()[ev].astype(np.float32),
        "gr_s21": gr_s.rolling(21, center=True, min_periods=1).std().fillna(0).to_numpy()[ev].astype(np.float32),
        "gr_d1": gr_s.diff().fillna(0).to_numpy()[ev].astype(np.float32),
        "frac": (np.arange(nev) / max(nev - 1, 1)).astype(np.float32),
    }
    for o in ANCH_OFF:
        f[f"twres_pf{o}"] = (gr_all[ev] - np.interp(pf[ev] + o, tw_tvt, tw_gr)).astype(np.float32)
    if is_train:
        f["target"] = (h["TVT"].to_numpy(float)[ev] - tps).astype(np.float32)
    return pd.DataFrame(f)




In [ ]:
import glob, numpy as np, pandas as pd, lightgbm as lgb, json
from scipy.signal import savgol_filter
from joblib import Parallel, delayed

def find_root():
    for r in ["/kaggle/input/rogii-wellbore-geology-prediction","data"]+sorted(glob.glob("/kaggle/input/*")):
        if os.path.exists(f"{r}/sample_submission.csv") and glob.glob(f"{r}/test/*__horizontal_well.csv"):
            return r
    hits=glob.glob("/kaggle/input/**/*__horizontal_well.csv",recursive=True)
    if hits: return os.path.dirname(os.path.dirname(hits[0]))
    raise FileNotFoundError("data")
DATA=find_root(); print("DATA:",DATA)

def mdir():
    hits=glob.glob("/kaggle/input/**/lgbm_stack.txt", recursive=True) or glob.glob("model_ds/lgbm_stack.txt")
    if not hits:
        raise FileNotFoundError("lgbm_stack.txt not found under /kaggle/input — is the model dataset attached? "+str(sorted(glob.glob("/kaggle/input/*"))))
    return os.path.dirname(hits[0])
MD_DIR=mdir(); print("MODEL:",MD_DIR)
booster=lgb.Booster(model_file=f"{MD_DIR}/lgbm_stack.txt")
meta=json.load(open(f"{MD_DIR}/meta.json")); FEATS=meta["feats"]; SH=meta["shrink"]; CL=meta["clip"]

paths=sorted(glob.glob(f"{DATA}/test/*__horizontal_well.csv"))
test=pd.concat([r for r in Parallel(n_jobs=4)(delayed(build_well)(p,False) for p in paths) if r is not None], ignore_index=True)
print("test signals", test.shape)

tp=np.clip(booster.predict(test[FEATS].to_numpy(np.float32))*SH,-CL,CL)
test=test.assign(pred_d=tp); rows=[]
for w,g in test.groupby("well",sort=False):
    v=g["pred_d"].to_numpy(float); n=len(v); wl=min(31, n if n%2==1 else n-1)
    if wl>=5: v=savgol_filter(v,wl,3)
    rows.append(pd.DataFrame({"id":g["id"].to_numpy(),"tvt":g["last_known_tvt"].to_numpy(float)+v}))
pred=pd.concat(rows,ignore_index=True)
sample=pd.read_csv(f"{DATA}/sample_submission.csv")
sub=sample[["id"]].merge(pred,on="id",how="left")
sub["tvt"]=sub["tvt"].fillna(test["last_known_tvt"].mean())
sub.to_csv("submission.csv",index=False)
print("wrote submission.csv", len(sub), "nan", sub["tvt"].isna().sum()); sub.head()